# GBL Score & Win Predictor

This notebook is a thin driver on top of the modules in [src/](src/). The
actual logic lives there — edit those files, not this notebook.

## Daily workflow

1. **Run the train cell.** It auto-syncs new results & upcoming fixtures from FlashScore into [data/raw/](data/raw/), then retrains on the fresh history. If the API is unreachable, the cell still trains using whatever's already on disk.
2. **Run the predict cell.** Click **Generate Predictions** to score the upcoming fixtures.

Team names in [data/raw/upcoming_fixtures.csv](data/raw/upcoming_fixtures.csv) must **exactly** match names in [data/raw/all_fixtures.csv](data/raw/all_fixtures.csv) — the pipeline validates this and raises with a `did you mean …?` suggestion if there's a typo.

The history of every prediction you've made (and how the model did) lives in [models_accuracy.ipynb](models_accuracy.ipynb).

If you change anything in `src/`, restart the kernel — Python caches modules after the first import.

In [7]:
# Install from requirements.txt — everything the project needs, pinned in one place.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Sync from FlashScore + retrain — one shot.
#
# 1. sync_for_notebook() pulls any new results into data/raw/all_fixtures.csv
#    and refreshes data/raw/upcoming_fixtures.csv. It NEVER raises — if the
#    API is down or the env vars aren't set, you'll see a warning here and
#    the rest of the notebook keeps working with whatever's already on disk.
# 2. train_and_save() then rebuilds features and refits on the (possibly
#    freshly-updated) history. Walk-forward metrics are honest because
#    sync only ADDS rows; it never reorders them.
from src.model import _format_metrics, train_and_save
from src.sync import sync_for_notebook

print(sync_for_notebook())
print()

bundle = train_and_save()
print(_format_metrics(bundle.metrics))

In [ ]:
# Upcoming fixtures + on-demand predictions. All the widget wiring lives
# in src/ui.py so this cell stays readable.
from src.ui import show_predictions_ui

show_predictions_ui(bundle)